#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

#Reading from bronze

In [0]:
df = spark.table("databricks_lakehouse.bronze.erp_px_cat_g1v2")
df.display()

#Transformation

##Trimming

In [0]:
trim_plan = {
    field.name : F.trim(F.col(field.name))
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
}
df = df.withColumns(trim_plan)

##Normalize Maintenance

In [0]:
df = df.withColumn("MAINTENANCE",
    F.when(F.col("MAINTENANCE")=="Yes", F.lit(True))
    .when(F.col("MAINTENANCE")=="No", F.lit(False))
    .otherwise(None)
)
df.display()

##Renaming columns

In [0]:
rename_plan = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag"
}
df = df.withColumnsRenamed(rename_plan)
df.limit(5).display()

#Writing in silver table

In [0]:
(
    df.write.mode("overwrite")
    .format("delta")
    .saveAsTable("databricks_lakehouse.silver.erp_product_catagery")
)

#Checking the silver table

In [0]:
%sql
select * from databricks_lakehouse.silver.erp_product_catagery limit 5